In [1]:
from qdrant_client import QdrantClient, models
from openai import OpenAI
import numpy as np
import uuid
from transformers import AutoTokenizer
from docling_core.types.doc import DoclingDocument
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from pathlib import Path

/home/pinch/projects/vLLM_demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EXPORT = Path('/home/pinch/projects/vLLM_demo/rag/demo/export_folder')

In [3]:
embedder_url = 'http://127.0.0.1:8001/v1'
qdrant_url = 'http://127.0.0.1:6333'
EMBEDDING_MODEL_NAME = 'bge-m3'


embedder = OpenAI(
  base_url=embedder_url,
  api_key='nokey'
)

qclient = QdrantClient(
  url=qdrant_url,
  api_key="1234"
)

/tmp/ipykernel_7215/3291360312.py:11: UserWarning: Api key is used with an insecure connection.
  qclient = QdrantClient(


In [4]:
tokenizer = HuggingFaceTokenizer(tokenizer=AutoTokenizer.from_pretrained('rag_models/embedder/bge-m3'), max_tokens=128)
chunker = HybridChunker(tokenizer=tokenizer)

In [5]:
def get_chunks(document_path, chunker):
    document = DoclingDocument.load_from_json(document_path)
    chunks = chunker.chunk(document)
    data = []
    for i, chunk in enumerate(chunks):
        data.append(
            {
                'chunk_id': i+1,
                'document': document_path.name,
                'text': chunker.contextualize(chunk)
            }
        )
    return data

In [6]:
def get_dense_vector(text, embedder):
  r = embedder.embeddings.create(model=EMBEDDING_MODEL_NAME, input=text)
  return np.array(r.data[0].embedding)


In [7]:
sparse_vector_options = {
  'language': 'french',
  'ascii_folding': True,
  'avg_len': 250
}

def get_sparse_vector(text):
  return models.Document(text=text, model='qdrant/bm25', options=sparse_vector_options)

In [8]:
def get_point_struct(chunk, embedder):
  text = chunk.get('text')
  chunk_id = chunk.get('chunk_id')
  file_path = chunk.get('document')
  
  payload = {
  'file_extension': 'pdf',
  'chunk_id': chunk_id,
  'file_path': file_path,
  'extraction_date': '2026-01-01',
  'text': text
  }

  vector = get_dense_vector(text, embedder)
  sparse_vector = get_sparse_vector(text)
  id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{chunk_id} - {file_path}'))

  point = models.PointStruct(
    id = id,
    vector = 
      {
        'dense': vector,
        'sparse': sparse_vector
      },
      payload = payload
  )
  
  return point
  

In [9]:
def get_points(chunks, embedder):
    return [get_point_struct(chunk, embedder) for chunk in chunks]    


In [10]:
collection_name = 'demo_collection'

if not qclient.collection_exists(collection_name):
  qclient.create_collection(
    collection_name,
    vectors_config={
      'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
      'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)
    } )


In [11]:
def get_all_chunks(docs):
  all_chunks = []
  for doc in docs:
    chunks = get_chunks(doc, chunker) 
    all_chunks.extend(chunks)
  return all_chunks

In [12]:
docs = sorted(EXPORT.glob('*.json'))
all_chunks = get_all_chunks(docs)
points = get_points(all_chunks, embedder)

qclient.create_payload_index(collection_name, field_name='file_path', field_schema=models.PayloadSchemaType.KEYWORD)
qclient.upsert(collection_name, points=points)

UpdateResult(operation_id=15, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
file_path = 'export_data/exemple.pdf'
chunk_id = 1

id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{chunk_id} - {file_path}'))
id

'65113525-6fa1-52f1-9b4c-fc7045515139'

In [14]:
collection_name = 'demo_collection'

if not qclient.collection_exists(collection_name):
  qclient.create_collection(
    collection_name,
    vectors_config={
      'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
      'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)
    } )

  qclient.update_collection(collection_name, 
                            )

In [15]:
qclient.create_payload_index(collection_name, field_name='file_path', field_schema=models.PayloadSchemaType.KEYWORD)

UpdateResult(operation_id=17, status=<UpdateStatus.COMPLETED: 'completed'>)

# EMBEDDING QUESTIONS

### Dense search

In [17]:
question = "Combien peux esperer gagner un employé de 5 ans d'expérience?"
r = embedder.embeddings.create(model=EMBEDDING_MODEL_NAME, input=question)
embedded_question = r.data[0].embedding

query_points = qclient.query_points(
  collection_name, using="dense",
  query = embedded_question, limit=10
)
query_points

QueryResponse(points=[ScoredPoint(id='64ec1b62-b3e9-58d0-b790-bd60d39a96cd', version=15, score=0.5532931, payload={'file_extension': 'pdf', 'chunk_id': 7, 'file_path': 'grille_salariale_2026.pdf.json', 'extraction_date': '2026-01-01', 'text': "1. Barème des salaires de base\n Salaire mensuel brut de base.Échelon 1 (0-2 ans) = 5 100 €. , Salaire mensuel brut de base.Échelon 2 (3-5 ans) = 5 420 €. , Salaire mensuel brut de base.Échelon 3 (6-10 ans) = 5 760 €. , Salaire mensuel brut de base.Échelon 4 (plus de 10 ans) = 6 120 €\nLe passage d'un échelon au suivant est automatique au 1er janvier qui suit l'anniversaire d'ancienneté correspondant."}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='bbdd30ab-eff4-5ca4-a9aa-a1a28511cf37', version=15, score=0.5488106, payload={'file_extension': 'pdf', 'chunk_id': 5, 'file_path': 'grille_salariale_2026.pdf.json', 'extraction_date': '2026-01-01', 'text': '1. Barème des salaires de base\n\n Catégorie Classe = D - Technicien expert. E

### Sparse search

In [18]:
sparse_query = models.Document(text=question, model='qdrant/bm25', options = sparse_vector_options)

qclient.query_points(
  collection_name,
  using='sparse',
  query= sparse_query
)

QueryResponse(points=[ScoredPoint(id='e8976781-5ff3-5224-992c-25b2091874aa', version=15, score=13.487623, payload={'file_extension': 'pdf', 'chunk_id': 4, 'file_path': 'grille_salariale_2026.pdf.json', 'extraction_date': '2026-01-01', 'text': '1. Barème des salaires de base\n\n assistant. Employés et employées, Salaire mensuel brut de base.Échelon 1 (0-2 ans) = 3 150 €. Employés et employées, Salaire mensuel brut de base.Échelon 2 (3-5 ans) = 3 310 €. Employés et employées, Salaire mensuel brut de base.Échelon 3 (6-10 ans) = 3 480 €. Employés et employées, Salaire mensuel brut de base.Échelon 4 (plus de 10 ans) = 3 660 €. Employés et employées,'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='bbdd30ab-eff4-5ca4-a9aa-a1a28511cf37', version=15, score=13.316778, payload={'file_extension': 'pdf', 'chunk_id': 5, 'file_path': 'grille_salariale_2026.pdf.json', 'extraction_date': '2026-01-01', 'text': '1. Barème des salaires de base\n\n Catégorie Classe = D - Technicien exper

### Hybrid Search == Reciprocal Rank Fusion (RRF)

In [19]:
search_results = qclient.query_points(
  collection_name,
  prefetch=[
    models.Prefetch(query=embedded_question, using="dense", limit=30),
    models.Prefetch(query=sparse_query, using="sparse", limit=30)
  ],
  query=models.RrfQuery(rrf=models.Rrf(k=60, weights=[1.3, 1])),
  limit=10
)


In [20]:
k=60
n = 10
dense_list_rank = 2
sparse_list_rank = 7

1/(1+k) + 1/(1+k)

0.03278688524590164

In [21]:
def build_context(search_results):
    blocks = []
    for i, p in enumerate(search_results.points, start=1):
        src = f"{p.payload['file_path']} (chunk {p.payload['chunk_id']})"
        blocks.append(f"[{i}] Source: {src}\n{p.payload['text']}")
    return "\n\n---\n\n".join(blocks)

context = build_context(search_results)

In [1]:
QWEN_URL = 'http://127.0.0.1:9931'
qwen_client = OpenAI(
  base_url=QWEN_URL,
  api_key='still_no_key'
)

sys_prompt = """Tu es un assistant qui répond aux questions à partir des documents fournis.
Règles :
- Réponds uniquement à partir du CONTEXTE ci-dessous. N'invente rien.
- Si le contexte ne contient pas la réponse, dis : "Je ne trouve pas cette information dans les documents."
- Cite le nom des documents utilisé comme sources.
- Réponds en français, de façon concise."""

user_prompt = f"""CONTEXTE :
{context}

QUESTION : {question}"""


r = qwen_client.chat.completions.create(
  model="local",
  messages=[
    {'role': 'system', 'content': sys_prompt},
    {'role': 'user', 'content': user_prompt },
  ],
    temperature=0.1,
    max_tokens=1024,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}}
)


NameError: name 'OpenAI' is not defined

In [24]:
print(r.choices[0].message)

ChatCompletionMessage(content="Le salaire exact dépend de la **catégorie** et de la **classe** de l'employé, car le barème varie selon ces critères. Voici les salaires mensuels bruts de base pour un employé avec 5 ans d'expérience (ce qui correspond généralement à l'**Échelon 2**, défini comme 3-5 ans) [1] [2] [4] [5] [6] :\n\n*   **Catégorie Employés :**\n    *   Classe C (Technicien) : **3 310 €** [1]\n    *   Classe D (Technicien expert) : **3 700 €** [2]\n    *   Classe E (Responsable d'équipe) : **4 450 €** [5]\n    *   Classe F (Responsable de service) : **5 420 €** [3]\n\n*   **Catégorie Ouvriers :**\n    *   Classe A (Opérateur) : **2 390 €** [6]\n    *   Classe B (Opérateur qualifié) : **2 670 €** [4]\n\nNote : Le passage d'échelon est automatique au 1er janvier suivant l'anniversaire d'ancienneté [3].", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [ ]:
def ask(question, limit=10):
    dense_q = embedder.embeddings.create(model=EMBEDDING_MODEL_NAME, input=question).data[0].embedding
    sparse_q = models.Document(text=question, model='qdrant/bm25', options=sparse_vector_options)
    res = qclient.query_points(
        collection_name,
        prefetch=[models.Prefetch(query=dense_q, using="dense", limit=30),
                  models.Prefetch(query=sparse_q, using="sparse", limit=30)],
        query=models.RrfQuery(rrf=models.Rrf(k=60, weights=[1.3, 1])),
        limit=limit,
    )
    r = qwen_client.chat.completions.create(
        model="local",
        messages=[{'role': 'system', 'content': sys_prompt},
                  {'role': 'user', 'content': f"CONTEXTE :\n{build_context(res)}\n\nQUESTION : {question}"}],
        temperature=0.1,
        max_tokens=1024,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}}
    )
    return r.choices[0].message.content, r.choices[0].finish_reason


In [ ]:




import polars as pl
from rag_questions import questions

for idx, q in enumerate(questions):
  q['answer'], q['finish_reason'] = ask(q['question'])
  q['expected_documents'] = ", ".join(q['expected_documents'])

df = pl.DataFrame(questions)
df.write_csv(f"FAQ.csv")

In [27]:
import csv
import os
from rag_questions import questions


fields = ["id", "question", "expected_documents", "answer", 'finish_reason']
done = set()

if os.path.exists("FAQ.csv"):
  with open("FAQ.csv", encoding="utf-8") as f:
    done = {row["id"] for row in csv.DictReader(f)}

with open("FAQ.csv", "w", newline="", encoding="utf-8") as f:
  writer = csv.DictWriter(f, fieldnames=fields)
  if not done:
    writer.writeheader()

  for q in questions:
    if q["id"] in done:
      continue

    answer = ask(q["question"])
    writer.writerow({
      "id": q["id"],
      "question": q["question"],
      "expected_documents": ", ".join(q["expected_documents"]),
      "answer": answer,
    })
    f.flush()   # push the row to disk now, not when the file closes
    print(f"{q['id']} done")

Q01 done
Q02 done
Q03 done
Q04 done
Q05 done
Q06 done
Q07 done
Q08 done
Q09 done
Q10 done
Q11 done
Q12 done
Q13 done
Q14 done
Q15 done
Q16 done
Q17 done
Q18 done
Q19 done
Q20 done
Q21 done
Q22 done
